# Forecast Insight Layer (Phase 5)

Run this on the machine with internet access and your API keys (your laptop).

It needs the forecast/backtest artifacts. Two ways to get them:
1. Copy these small files from the server into the matching folders here:
   `reports/forecast/series_context.parquet`, `reports/forecast/forecast.parquet`,
   `reports/backtests/leaderboard.csv`, `reports/backtests/lightgbm_feature_importance.csv`,
   `models/lightgbm_best_params.json`.
2. Or run the optional export cell near the bottom to regenerate them locally.

Keys are entered securely with getpass and are never written to disk.

## 1. Locate the project

In [ ]:
import os, sys, glob
# Adjust if your notebook is not already in the project root:
# os.chdir(r'D:\PROJECTS\TIME1')
print('cwd:', os.getcwd())
assert os.path.isdir('src'), 'Run from the project root (the folder containing src/).'

have = {
    'leaderboard': os.path.exists('reports/backtests/leaderboard.csv'),
    'feature_importance': os.path.exists('reports/backtests/lightgbm_feature_importance.csv'),
    'series_context': os.path.exists('reports/forecast/series_context.parquet'),
    'best_params': os.path.exists('models/lightgbm_best_params.json'),
}
print('artifacts present:', have)
if not have['series_context']:
    print('\nseries_context missing -> copy artifacts from the server, or run the optional export cell below.')

## 2. Install the insight dependencies

In [ ]:
!{sys.executable} -m pip install -q -r requirements-insight.txt

## 3. Enter your API keys (secure, not saved)

Leave a field blank to skip that provider. You need at least one.

In [ ]:
import getpass
g = getpass.getpass('Groq API key (blank to skip): ')
if g: os.environ['GROQ_API_KEY'] = g
m = getpass.getpass('Gemini API key (blank to skip): ')
if m: os.environ['GEMINI_API_KEY'] = m
print('Groq set:', bool(os.environ.get('GROQ_API_KEY')), '| Gemini set:', bool(os.environ.get('GEMINI_API_KEY')))

## 4. Build the analyst (rotating provider by default)

In [ ]:
from src.config import load_config
from src.insight.analyst import build_analyst

cfg = load_config()
analyst = build_analyst(cfg)   # uses insight.provider from config (rotating)
print('provider chain:', analyst.provider.name)

## 5. Executive summary

In [ ]:
print(analyst.executive_summary())

## 6. Ask a grounded question (edit the text)

In [ ]:
question = 'Which category carries the most forecast risk, and why?'
print(analyst.answer(question))

## 7. Explain one series (auto-picks the highest-volume one)

In [ ]:
import pandas as pd
sc = pd.read_parquet('reports/forecast/series_context.parquet')
series_id = sc.sort_values('forecast_total', ascending=False)['id'].iloc[0]
print('Explaining:', series_id, '\n')
print(analyst.explain_series(series_id))

## Optional: regenerate artifacts locally

Only run this if you did NOT copy the artifacts from the server. It fits the
tuned LightGBM on all data and can take several minutes and a few GB of RAM.
Copy `models/lightgbm_best_params.json` from the server first to use the tuned
hyperparameters; otherwise it falls back to defaults.

In [ ]:
!{sys.executable} -u -m scripts.export_forecast